In [15]:
# ==============================================================================
# Project: Al-Rafidain Platform - Predictive Diagnosis System
# Company: Quality Pioneers (Ruwad Al-Joudah)
# Developer/Presenter: Fatima Muhannad Salah (College of Medicine, Ashur University)
# Description: This code integrates Google Earth Engine with advanced Computer Vision.
# ==============================================================================

import ee
import geemap
import cv2
import numpy as np

# Authentication and Initialization (الربط مع خوادم جوجل السحابية)
# يجب وضع مُعرّف المشروع الخاص بك هنا لضمان نجاح الاتصال
PROJECT_ID = 'sincere-lexicon-493313-e7'

try:
    # محاولة التهيئة باستخدام معرّف المشروع
    ee.Initialize(project=PROJECT_ID)
    print("[INFO] Google Earth Engine Initialized Successfully.")
except Exception as e:
    print("[WARNING] Authentication Required. Please follow the link to authenticate.")
    # طلب المصادقة إذا لم تكن مفعلة
    ee.Authenticate()
    # إعادة التهيئة مع معرّف المشروع بعد المصادقة
    ee.Initialize(project=PROJECT_ID)
    print("[INFO] Authentication Successful.")

[INFO] Google Earth Engine Initialized Successfully.


In [14]:
import ee
ee.Authenticate()
ee.Initialize(project='sincere-lexicon-493313-e7')
print("✅ الاتصال بالأقمار الصناعية نشط وجاهز للتحليل.")

✅ الاتصال بالأقمار الصناعية نشط وجاهز للتحليل.


In [ ]:
# 3. كتابة ملف المنصة الرئيسي (app.py) النسخة النهائية بتنسيق الأسماء الموحد
with open('app.py', 'w') as f:
    f.write("""
import streamlit as st
import cv2
import numpy as np
import datetime
import ee
import os
import glob
from PIL import Image

# إعدادات الصفحة
st.set_page_config(page_title="بوابة الرافدين الذكية ", layout="wide")

# --- الهوية المؤسسية ---
logo_files = glob.glob('logo.*') + glob.glob('Logo.*')
if logo_files:
    try: st.sidebar.image(Image.open(logo_files[0]), use_container_width=True)
    except: st.sidebar.title("🛡️ كلية الطب العام-جامعة بغداد")
else: st.sidebar.title("🛡️ ")

st.sidebar.markdown("---")

# --- لوحة التحكم ---
sector = st.sidebar.radio("اختر المحور الاستراتيجي للتحليل:",
                         ["الرؤية العامة", "القطاع الزراعي", "القطاع النفطي", "قطاع المسح الجيولوجي والتعدين", "حماية الثروة السمكية", "التشخيص الطبي المرجعي"])

st.sidebar.markdown("---")

# إعدادات الجيومكانية
if sector in ["الرؤية العامة", "القطاع الزراعي", "القطاع النفطي", "قطاع المسح الجيولوجي والتعدين", "حماية الثروة السمكية"]:
    st.sidebar.subheader("📍 إعدادات النطاق المساحي")
    lat = st.sidebar.number_input("خط العرض (Lat):", value=36.3486, format="%.6f")
    lon = st.sidebar.number_input("خط الطول (Lon):", value=43.1267, format="%.6f")
    buffer_size = st.sidebar.slider("قطر منطقة المسح (متر):", 100, 5000, 1000)
    st.sidebar.markdown("---")
    pid = st.sidebar.text_input("Project ID:", value="sincere-lexicon-493313-e7")
    try:
        ee.Initialize(project=pid)
        connected = True
        st.sidebar.success("الاتصال الفضائي نشط ✅")
    except: connected = False

if sector == "التشخيص الطبي المرجعي":
    st.sidebar.subheader("🏥 معايرة الحساسية الطبية")
    sens = st.sidebar.slider("عتبة العزل الرقمي للنسيج:", 50, 255, 140)
    limit = st.sidebar.slider("حد الخطر السريري (Opacity %):", 1.0, 30.0, 5.0)

# محرك معالجة البيانات الفضائية
def perform_deep_analysis(lt, ln, size, s_name):
    point = ee.Geometry.Point([ln, lt])
    area = point.buffer(size/2).bounds()
    img = ee.ImageCollection('COPERNICUS/S2_SR').filterBounds(area).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 15)).sort('system:time_start', False).first()
    stats = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=area, scale=10).getInfo()
    d_str = datetime.datetime.fromtimestamp(img.get('system:time_start').getInfo()/1000.0).strftime('%Y-%m-%d')

    ndvi = (stats['B8'] - stats['B4']) / (stats['B8'] + stats['B4']) if 'B8' in stats else 0
    ndwi = (stats['B3'] - stats['B8']) / (stats['B3'] + stats['B8']) if 'B3' in stats else 0
    iron = stats['B4'] / stats['B2'] if 'B2' in stats else 0
    clay = stats['B11'] / stats['B12'] if 'B12' in stats else 0
    thermal = stats['B12'] if 'B12' in stats else 0

    if s_name == "القطاع الزراعي": p = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
    elif s_name == "حماية الثروة السمكية": p = {'bands': ['B8', 'B3', 'B2'], 'min': 0, 'max': 3000}
    elif s_name == "القطاع النفطي": p = {'bands': ['B12', 'B8A', 'B4'], 'min': 0, 'max': 4000}
    else: p = {'bands': ['B11', 'B12', 'B4'], 'min': 0, 'max': 5000}
    return img.getThumbURL({**p, 'region': area, 'dimensions': 800, 'format': 'png'}), d_str, ndvi, ndwi, iron, clay, thermal

# --- العرض الرئيسي والتقارير التفصيلية المعمقة ---
if sector == "الرؤية العامة":
    st.markdown("<h2 style='text-align: center; color: #1E3A8A; margin-bottom: 0;'>بوابة الرافدين الذكية - النموذج الاولي للمنتج    </h2>", unsafe_allow_html=True)
    st.markdown("<h2 style='text-align: center; color: #1E3A8A; margin-bottom: 0;'> كلية الطب البشري - جـامـعـة آشـور  </h2>", unsafe_allow_html=True)
    st.markdown("<h4 style='text-align: center; color: #4B5563; margin-top: 10px;'>هذه المنصة تثبت قدرة الخوارزمية على تشخيص حالات مختلفة تماماً بنفس المنطق البرمجي</h4>", unsafe_allow_html=True)
    st.markdown(\"\"\"
    <div style='text-align: center; margin-top: 30px;'>
        <h3 style='color: #1E3A8A;'>   فـاطـمـة مهنـد صـلاح</h3>
        <h3 style='color: #1E3A8A;'>   طالبة في كلية الطب </h3>
         <h3 style='color: #1E3A8A;'> " الخوارزمية التي ستحمي نفط العراق وبيئته وأمنه الغذائي ، ولدت من رحم الخوارزمية التي ستحمي صحة العراقيين" </h3>
    </div>
    \"\"\", unsafe_allow_html=True)
    st.markdown("---")
    st.info("مرحباً بك في النظام الاستشاري الخبير. اختر القطاع من القائمة الجانبية لبدء التحليل الدقيق.")

elif sector in ["القطاع الزراعي", "القطاع النفطي", "قطاع المسح الجيولوجي والتعدين", "حماية الثروة السمكية"] and connected:
    url, date, ndvi, ndwi, iron, clay, thermal = perform_deep_analysis(lat, lon, buffer_size, sector)
    c1, c2 = st.columns([2, 1])
    with c1: st.image(url, caption=f"رصد فضائي متقدم للمساحة المحددة - {date}", use_container_width=True)
    with c2:
        st.write(f"### 📑 التقرير الاستشاري المعمق - {sector}")

        if sector == "القطاع الزراعي":
            st.metric("مؤشر الكتلة الحيوية (NDVI)", f"{ndvi:.2f}")
            with st.expander("🩺 التحليل الفسيولوجي والمسببات", expanded=True):
                if ndvi < 0.45:
                    st.error("🚨 التشخيص: إجهاد كلوروفيلي حاد (Severe Chlorosis) وضعف في العمليات الأيضية.")
                    st.markdown(\"\"\"
                    **التحليل العلمي والمسببات:**
                    * **نقص العناصر الكبرى:** تراجع حاد في معدلات النيتروجين (N) والفسفور (P)، مما يعيق بناء الأحماض الأمينية وانقسام الخلايا.
                    * **الإجهاد الأسموزي (Osmotic Stress):** ارتفاع قراءات ملوحة التربة يؤدي إلى "العطش الفسيولوجي"، حيث تفقد الجذور قدرتها على سحب المياه، مما يسبب انغلاق الثغور الورقية (Stomata).
                    * **ضعف المجموع الجذري:** عدم قدرة الجذور على التمدد الفعال لاقتناص المغذيات من طبقات التربة العميقة.
                    \"\"\")
                else:
                    st.success("✅ التشخيص: نمو نباتي مثالي وكفاءة عالية في التمثيل الضوئي.")
                    st.write("**التفسير:** قراءات الانعكاس الطيفي للأشعة تحت الحمراء القريبة (NIR) تؤكد صحة الجدار الخلوي ووفرة الصبغات النباتية.")
            with st.expander("💊 البروتوكول العلاجي والتوصيات", expanded=True):
                if ndvi < 0.45:
                    st.markdown(\"\"\"
                    **خطة التدخل والهندسة الزراعية:**
                    1. **العلاج الموجه:** المباشرة الفورية بحقن أو رش المخصب النانوي **Super Bio Nano**. الجزيئات النانوية تخترق الأنسجة بسرعة فائقة لتعويض النقص خلال 48 ساعة.
                    2. **معالجة التربة:** استخدام محسنات التربة (Soil Conditioners) لفك ارتباط الأملاح وتعديل الـ pH لضمان تحرير العناصر الصغرى المحتبسة.
                    3. **الإدارة المائية:** تطبيق دورات ري تقنينية (Deficit Irrigation) لغسيل الأملاح بعيداً عن منطقة الجذور الفعالة.
                    \"\"\")
                else: st.write("التوصية: الاستمرار في البرنامج المتبع وإجراء مسح دوري كل 14 يوماً.")

        elif sector == "حماية الثروة السمكية":
            st.metric("مؤشر البيئة المائية (NDWI)", f"{ndwi:.2f}")
            with st.expander("🩺 التقييم الهيدرولوجي والمسببات", expanded=True):
                if ndwi < 0.15:
                    st.error("🚨 التشخيص: تدهور خطير في جودة المسطح المائي واحتمالية نقص التأكسج (Hypoxia).")
                    st.markdown(\"\"\"
                    **التحليل البيئي والمسببات:**
                    * **العكارة (Turbidity):** تراكم عالٍ للمواد العضوية المعلقة يمنع اختراق أشعة الشمس، مما يعيق عملية البناء الضوئي للعوالق النباتية (Phytoplankton).
                    * **السمية الكيميائية:** التحلل اللاهوائي للفضلات قد يؤدي لارتفاع مستويات الأمونيا السامة (NH3) وكبريتيد الهيدروجين في القاع.
                    * **الركود المائي:** غياب التيارات المائية يسبب تطبقاً حرارياً (Thermal Stratification) يحبس الأكسجين في الطبقة السطحية فقط.
                    \"\"\")
                else:
                    st.success("✅ التشخيص: استقرار كيميائي وفيزيائي للبيئة المائية.")
                    st.write("**التفسير:** التوازن الأمثل بين نفاذية الضوء والمغذيات، مما يدعم دورة حياة آمنة للأسماك.")
            with st.expander("💊 بروتوكول المعالجة المائية", expanded=True):
                if ndwi < 0.15:
                    st.markdown(\"\"\"
                    **خطة الطوارئ البيئية:**
                    1. **التهوية القسرية:** تشغيل البدالات الميكانيكية (Paddlewheels) فوراً لكسر التطبق الحراري ورفع تركيز الأكسجين المذاب (DO).
                    2. **المعالجة الحيوية:** إضافة بكتيريا نافعة (Probiotics) لتحليل الرواسب العضوية في القاع وتقليل نسبة الأمونيا.
                    3. **الإدارة التشغيلية:** إيقاف التغذية (الأعلاف) مؤقتاً لحين استقرار المعايير، وتفريغ 20% من المياه القاعية واستبدالها بمياه جديدة.
                    \"\"\")
                else: st.write("التوصية: المحافظة على معدلات التهوية الحالية ومراقبة مستويات العلف.")

        elif sector == "قطاع المسح الجيولوجي والتعدين":
            st.write(f"مؤشر الانعكاس الفلزي (Iron): **{iron*10:.2f}%** | مؤشر اللامعادن (Clay): **{clay*15:.2f}%**")
            with st.expander("🩺 التحليل الطيفي المعدني", expanded=True):
                st.info("الاستنتاج: رصد تباين طيفي يشير لتكوينات رسوبية غنية بالخامات الصناعية.")
                st.markdown(\"\"\"
                **التفسير الجيولوجي:**
                * **الأكاسيد (Iron):** يعكس التحليل وجود خامات الهيماتيت/الجويثيت، الناتجة عن التجوية الكيميائية الطويلة الأمد للصخور الأصلية.
                * **السيليكات (Clay):** الامتصاص العالي في نطاق الأشعة تحت الحمراء قصيرة الموجة (SWIR) يؤكد وجود معادن الكاولين والجبس، وهي ذات قيمة اقتصادية عالية في صناعة الأسمنت والحراريات.
                \"\"\")
            with st.expander("💊 الجدوى وخطة الاستخراج", expanded=True):
                st.markdown(\"\"\"
                **التوصيات الهندسية:**
                1. المباشرة بحفر آبار استكشافية لاستخراج عينات لبية (Core Samples).
                2. إجراء تحاليل حيود الأشعة السينية (XRD) لتأكيد نقاوة الأطوار المعدنية.
                3. إعداد دراسة جدوى فنية للانتقال إلى مرحلة التعدين السطحي المفتوح (Open-pit mining).
                \"\"\")

        elif sector == "القطاع النفطي":
            st.metric("البصمة الديناميكية الحرارية", f"{thermal:.0f}")
            with st.expander("🩺 التقييم الأمني والتشخيص", expanded=True):
                if thermal > 3850:
                    st.error("🚨 تحذير بالغ: رصد شذوذ حراري (Thermal Anomaly) يتجاوز العتبة الآمنة.")
                    st.markdown(\"\"\"
                    **التحليل والمسببات:**
                    * **إجهاد ميكانيكي:** احتكاك عالٍ أو انخفاض حاد في الضغط داخل الصمامات (Joule-Thomson Effect) يولد بؤرة حرارية.
                    * **تسرب هيدروكربوني:** احتمالية وجود تسرب نفطي ميكروي يتفاعل مع أكسجين الهواء مسبباً انبعاثاً حرارياً تحت السطح (Incipient Fire).
                    \"\"\")
                else:
                    st.success("✅ التشخيص: استقرار تام في البنية التحتية.")
                    st.write("التفسير: الانبعاث الحراري ضمن النطاقات المعيارية لخطوط النقل والمنشآت.")
            with st.expander("💊 بروتوكول الاستجابة", expanded=True):
                if thermal > 3850:
                    st.markdown(\"\"\"
                    **الإجراءات الفورية:**
                    1. توجيه طائرات مسيرة مزودة بكاميرات تصوير حراري (IR Drones) للتدقيق الموقعي الدقيق.
                    2. إغلاق الصمامات المقطعية لعزل المنطقة المتضررة احترازياً.
                    3. إجراء فحص ضغط هيدروستاتيكي للأنبوب المستهدف للتأكد من سلامة اللحامات.
                    \"\"\")
                else: st.write("التوصية: استمرار دوريات المراقبة الفضائية المبرمجة.")

# --- المحور الطبي المرجعي (استشاري مفصل) ---
elif sector == "التشخيص الطبي المرجعي":
    st.header("🏥 وحدة التشخيص الشعاعي الخبير (Radiology Expert AI)")
    st.markdown("تحليل نسيجي فائق الدقة لصور الأشعة السينية (X-Ray) والمقطعية (CT) لكشف أنماط العتامة والارتشاح.")

    up = st.file_uploader("قم برفع صورة الأشعة الطبية هنا:", type=["jpg", "png", "jpeg"])

    if up:
        img_pil = Image.open(up).convert('RGB')
        img_cv = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
        gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=3.5, tileGridSize=(8,8)); enhanced = clahe.apply(gray)
        _, th = cv2.threshold(enhanced, sens, 255, cv2.THRESH_BINARY)
        opac = (np.sum(th == 255) / th.size) * 100
        heatmap = cv2.applyColorMap(enhanced, cv2.COLORMAP_JET)

        c1, c2, c3 = st.columns(3)
        c1.image(img_pil, caption="1. الصورة السريرية الأصلية", use_container_width=True)
        c2.image(cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB), caption="2. الخارطة الحرارية لتوزيع الكثافة", use_container_width=True)
        c3.image(th, caption="3. قناع العزل الرقمي (العتامة)", use_container_width=True)

        st.markdown("---")
        st.subheader("📋 التقرير الطبي الاستشاري الشامل (Clinical Report)")
        res1, res2 = st.columns(2)

        with res1:
            st.write("### 🔍 التقييم الشعاعي والمسببات")
            st.metric("معدل الكثافة النسيجية (Tissue Opacity)", f"{opac:.2f}%")
            if opac > limit:
                st.error("🚨 التشخيص المبدئي: رصد أنماط 'ارتشاح رئوي' (Pulmonary Infiltration) وتكثف سنخي (Alveolar Consolidation).")
                st.markdown(\"\"\"
                **التفسير الباثولوجي (المسببات):**
                * استبدال الهواء الطبيعي في الحويصلات الهوائية بسوائل ارتشاحية (Exudate) أو قيح ناتج عن التهاب بكتيري/فيروسي (Pneumonia).
                * احتمالية وجود وذمة رئوية (Pulmonary Edema) ناتجة عن احتقان الأوعية الدموية.
                * الخارطة الحرارية (باللون الأحمر) تشير بدقة إلى تمركز هذه الكثافات بشكل غير متجانس في الحقول الرئوية.
                \"\"\")
            else:
                st.success("✅ التشخيص المبدئي: الحقول الرئوية واضحة وسليمة (Clear Lung Fields).")
                st.markdown(\"\"\"
                **التفسير:** * نفاذية الأشعة (Radiolucency) ممتازة وتدل على امتلاء الحويصلات بالهواء بشكل طبيعي.
                * لا توجد بؤر تكثف (Consolidations) أو عقد نسيجية غير طبيعية تتجاوز الحدود المرجعية.
                \"\"\")

        with res2:
            st.write("### 💊 البروتوكول السريري المقترح")
            if opac > limit:
                st.warning("**خطة الإدارة السريرية والتشخيص التفريقي:**")
                st.markdown(\"\"\"
                1. **التصوير المتقدم:** طلب صورة مقطعية عالية الدقة (HRCT) لتحديد التوزيع المورفولوجي للارتشاح بشكل ثلاثي الأبعاد.
                2. **التحاليل المخبرية:** سحب عينات دم عاجلة لفحص مؤشرات الالتهاب (CRP, Procalcitonin, CBC) وإجراء زراعة بلغم (Sputum Culture) لتحديد المسبب الميكروبي.
                3. **التدخل العلاجي:** البدء الفوري بإعطاء مضادات حيوية واسعة الطيف (Empiric Antibiotics) وريدياً لحين ظهور نتائج المزرعة.
                4. **المراقبة الحيوية:** الربط المستمر بجهاز مراقبة تشبع الأكسجين في الدم (SpO2) وتقديم الدعم التنفسي (Oxygen Therapy) إذا انخفضت النسبة عن 92%.
                \"\"\")
            else:
                st.info("**توصيات المتابعة:**")
                st.markdown(\"\"\"
                * لا حاجة لأي تداخل دوائي أو إشعاعي إضافي في الوقت الحالي.
                * يُنصح بالمتابعة السريرية الروتينية بعد 6 أشهر، أو المراجعة الفورية في حال ظهور أعراض تنفسية حادة مثل السعال المستمر أو ضيق التنفس.
                \"\"\")
""")

In [16]:
# 4. تشغيل المنصة وفتح نافذة العرض
!fuser -k 8501/tcp
import os, time
from google.colab import output
os.system("streamlit run app.py --server.port 8501 --server.enableCORS false &")
time.sleep(10)
output.serve_kernel_port_as_iframe(8501, height=850)

8501/tcp:             4000


<IPython.core.display.Javascript object>